# Server-side classification evaluation (template)

This notebook is a **template** for comparing models on the `ClassificationCategory` / `ClassificationModel` stack.

**Avoid running model cells on a laptop** if you execute elsewhere: `HF_LLM`, `Ollama_LLM`, etc. **download weights** or open network services. Set `enabled=True` only in `MODEL_SPECS` on the machine that should load models.

Categories are read from JSON under `src/nps_crawling/classification/configurations/categories` (layout follows `Config.CLASSIFICATION_CONFIG_USE_NAME_FILES`).


In [ ]:
%pip install scikit-learn
%pip install pandas
%pip install -e ..
%pip install accelerate
%pip install transformers
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

## 1. Discover categories from configuration files

`discover_categories()` loads each category JSON from the packaged tree (skips `Default` / `default`).

Set `CATEGORY_CONFIG_PATHS` to a non-empty list of `Path` objects to load **only** those files (absolute paths or paths relative to the notebook cwd).


In [3]:
from __future__ import annotations

import json

from nps_crawling.config import Config
from nps_crawling.classification.categories.category import ClassificationCategory
import json
from nps_crawling.config import Config

nps_category_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Category" / "bf600907ffcecc0de9cc15ae509838b35e2468c3f03635daa12424566819f689.json"
nps_value_category_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Category" / "b42b037d61791e938c8df7ad6e4d9c4e06069004c9386deae246fa0e5e4b5003.json"
nps_numeric_nps_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Category" / "e8744899b197b6f0468e52bf97754564157c1c112a422ec9381d6ea66ff54d4f.json"
nps_all_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Category" / "5d15d7f5de353fd44805fc06bbe9432da4990d2524bf44cf40a826060711cb24.json"

with open(nps_category_file, "r") as f:
    data = json.load(f)
    nps_category = ClassificationCategory.from_dict(data)
with open(nps_category_file, "r") as f:
    data = json.load(f)
    nps_value_category = ClassificationCategory.from_dict(data)
with open(nps_category_file, "r") as f:
    data = json.load(f)
    has_numeric_nps = ClassificationCategory.from_dict(data)
with open(nps_category_file, "r") as f:
    data = json.load(f)
    nps_all = ClassificationCategory.from_dict(data)

## 2. Model instances (edit `MODEL_SPECS` on the runner machine)

**Option A — `get_model`:** use `ClassificationModelName` + `get_model(...)` (kwargs forwarded to the concrete class).

**Option B — direct classes:** import `HF_LLM`, `OpenAIModel`, `Ollama_LLM`, … and construct them explicitly (same as the registry does internally).

Set `enabled=True` for specs you want `build_models()` to instantiate.


In [4]:
import os
with open(r"C:\Users\WINTER_S\Documents\Master\openai_token.txt", "r") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key

In [5]:
from nps_crawling.classification.models.registry import ClassificationModelName, get_model

model = get_model(ClassificationModelName.OPENAI, "gpt-5.4-mini")

c:\Users\WINTER_S\Documents\Master\NPS-Crawling\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Evaluation grid

`ClassificationModel.evaluate(category, text_column, test_size=None)` reads the ground-truth CSV from `category.csv_path` (relative paths are anchored at `Config.ROOT_DIR` in the library). Metrics are written back to each model's JSON config under `evaluation_results`.


In [6]:
model.evaluate(nps_all, test_size=0.02)

APIConnectionError: Connection error.